In [ ]:
import os
import math
import argparse
from dataclasses import dataclass
from typing import Tuple, List, Optional
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, utils as vutils
import matplotlib.pyplot as plt

# This is used for dedicated GPU utilization, default it automatically fetch CUDA instead default tensor GPU it will utilize

def get_device():
    """Get the best available device (CUDA GPU if available, else CPU)"""
    if torch.cuda.is_available():
        device = torch.device("cuda")
        # Print GPU information
        gpu_name = torch.cuda.get_device_name(0)
        gpu_count = torch.cuda.device_count()
        print(f"   CUDA is available! Using GPU: {gpu_name}")
        print(f"   Number of GPUs: {gpu_count}")
        print(f"   Current device: {torch.cuda.current_device()}")
        print(f"   Device capability: {torch.cuda.get_device_capability(0)}")

        # Set CUDA to use the best algorithm
        torch.backends.cudnn.benchmark = True
        torch.backends.cudnn.deterministic = False
    else:
        device = torch.device("cpu")
        print("CUDA is not available. Using CPU (training will be slower)")

    return device

# Set device globally
DEVICE = get_device()

# Task 1: Architectural Design — Encoder maps RGB image -> (mu, logvar)
class Encoder(nn.Module):
    def __init__(self, latent_dim: int = 64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1),  # 32x32 -> 16x16
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),  # 16x16 -> 8x8
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),  # 8x8 -> 4x4
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.Conv2d(128, 256, kernel_size=4, stride=1, padding=0),  # 4x4 -> 1x1
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        self.flatten = nn.Flatten()
        self.fc_mu = nn.Linear(256 * 1 * 1, latent_dim)
        self.fc_logvar = nn.Linear(256 * 1 * 1, latent_dim)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        h = self.conv(x)
        h = self.flatten(h)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar


# Task 1: Architectural Design — Decoder maps latent z -> reconstructed RGB image
class Decoder(nn.Module):
    def __init__(self, latent_dim: int = 64):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 256 * 1 * 1)
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=1, padding=0),  # 1x1 -> 4x4
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),  # 4x4 -> 8x8
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),  # 8x8 -> 16x16
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1),  # 16x16 -> 32x32
            nn.Sigmoid(),  # outputs in [0,1]
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        h = self.fc(z)
        h = h.view(-1, 256, 1, 1)
        x_hat = self.deconv(h)
        return x_hat


class VAE(nn.Module):
    def __init__(self, latent_dim: int = 64):
        super().__init__()
        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(latent_dim)
        self.latent_dim = latent_dim

    # Task 1: Bridge — Reparameterization trick (z = mu + sigma * epsilon)
    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decoder(z)
        return x_hat, mu, logvar

# Loss and training utilities

@dataclass
class TrainConfig:
    epochs: int = 5 #50
    batch_size: int = 128
    lr: float = 1e-3
    latent_dim: int = 64
    beta: float = 1.0
    num_workers: int = 2
    device: torch.device = DEVICE
    out_dir: str = "outputs"
    seed: int = 42


def set_seed(seed: int = 42):
    """Set all random seeds for reproducibility"""
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)
    print(f"🔐 Random seed set to {seed}")


# Task 4: β-VAE Modification — Loss = Recon + β * KL
def vae_loss(x: torch.Tensor, x_hat: torch.Tensor, mu: torch.Tensor, logvar: torch.Tensor, beta: float = 1.0):
    recon = F.binary_cross_entropy(x_hat, x, reduction='sum') / x.size(0)
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
    loss = recon + beta * kl
    return loss, recon, kl


def build_dataloaders(batch_size: int, num_workers: int = 2, device: torch.device = None):
    """Build dataloaders with appropriate settings based on device"""
    transform = transforms.Compose([
        transforms.ToTensor(),
    ])

    train_ds = datasets.CIFAR10(root="./data", train=True, transform=transform, download=True)
    test_ds = datasets.CIFAR10(root="./data", train=False, transform=transform, download=True)

    # Only use pin_memory if using CUDA
    use_pin_memory = device.type == 'cuda' if device else torch.cuda.is_available()

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=use_pin_memory
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=use_pin_memory
    )

    if use_pin_memory:
        print(f"📦 Using pin_memory for faster GPU transfer")

    return train_loader, test_loader


def save_image_grid(tensor: torch.Tensor, nrow: int, path: str, title: str = None):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    grid = vutils.make_grid(tensor, nrow=nrow, padding=2, normalize=False)
    plt.figure(figsize=(12, 12))
    if title:
        plt.title(title, fontsize=16)
    plt.axis('off')
    plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
    plt.tight_layout()
    plt.savefig(path, bbox_inches='tight', pad_inches=0.1, dpi=150)
    plt.close()


def plot_curves(train_losses, recon_losses, kl_losses, path: str, beta: float):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Total Loss', linewidth=2)
    plt.plot(recon_losses, label='Reconstruction Loss', linewidth=2)
    plt.plot(kl_losses, label=f'KL Loss (β={beta})', linewidth=2)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.title(f'Training Curves (β={beta})', fontsize=14)
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()


# Generating 4x4 grid of samples (random z ~ N(0, I))
def sample_random_grid(model: VAE, device: torch.device, latent_dim: int, out_path: str, n: int = 16):
    model.eval()
    with torch.no_grad():
        z = torch.randn(n, latent_dim, device=device)
        samples = model.decoder(z).cpu()
    save_image_grid(samples, nrow=int(math.sqrt(n)), path=out_path,
                   title=f"Random Samples (β={getattr(model, 'beta', 1.0)})")


# Latent Space Interpolation — 10-step linear morph between two real images
def interpolate_latents(model: VAE, device: torch.device, latent_dim: int, out_path: str,
                        steps: int = 10, loader: DataLoader = None):
    model.eval()
    with torch.no_grad():
        if loader is not None:
            # Taking random real images
            data_iter = iter(loader)
            images, _ = next(data_iter)
            # Move images to device
            images = images.to(device)
            # Select two random images
            idx1, idx2 = np.random.choice(len(images), 2, replace=False)
            img1, img2 = images[idx1].unsqueeze(0), images[idx2].unsqueeze(0)

            # Encode to get latent representations
            mu1, logvar1 = model.encoder(img1)
            mu2, logvar2 = model.encoder(img2)

            # Use means for interpolation
            z1, z2 = mu1, mu2
        else:
            # Fallback to random latent vectors
            z1 = torch.randn(1, latent_dim, device=device)
            z2 = torch.randn(1, latent_dim, device=device)

        imgs = []
        alphas = []
        for i in range(steps):
            t = i / (steps - 1)
            zt = (1 - t) * z1 + t * z2
            x_hat = model.decoder(zt).cpu()
            imgs.append(x_hat[0])
            alphas.append(f"{t:.1f}")

        imgs = torch.stack(imgs, dim=0)

        # representing informative visualization
        fig, axes = plt.subplots(1, steps, figsize=(20, 2.5))
        for i, (img, alpha) in enumerate(zip(imgs, alphas)):
            axes[i].imshow(img.permute(1, 2, 0).numpy())
            axes[i].set_title(f"α={alpha}", fontsize=10)
            axes[i].axis('off')
        plt.suptitle(f"Latent Space Interpolation (β={getattr(model, 'beta', 1.0)})", fontsize=14, y=1.05)
        plt.tight_layout()
        plt.savefig(out_path, dpi=150, bbox_inches='tight')
        plt.close()


def plot_reconstructions(model: VAE, loader: DataLoader, device: torch.device, out_path: str, num_images: int = 8):
    """Plot original vs reconstructed images"""
    model.eval()
    with torch.no_grad():
        # Bulk set of Image batch
        data_iter = iter(loader)
        images, _ = next(data_iter)
        images = images[:num_images].to(device)

        # Get Image reconstructions
        x_hat, _, _ = model(images)

        # Combining originals and reconstructions
        comparison = torch.cat([images.cpu(), x_hat.cpu()], dim=0)

        fig, axes = plt.subplots(2, num_images, figsize=(15, 4))
        for i in range(num_images):
            # Original
            axes[0, i].imshow(images[i].cpu().permute(1, 2, 0))
            axes[0, i].axis('off')
            if i == 0:
                axes[0, i].set_ylabel('Original', fontsize=12)

            # Reconstruction
            axes[1, i].imshow(x_hat[i].cpu().permute(1, 2, 0))
            axes[1, i].axis('off')
            if i == 0:
                axes[1, i].set_ylabel('Reconstructed', fontsize=12)

        plt.suptitle(f"Original vs Reconstructed Images (β={getattr(model, 'beta', 1.0)})", fontsize=14)
        plt.tight_layout()
        plt.savefig(out_path, dpi=150, bbox_inches='tight')
        plt.close()


# Training and Performance — optimize Recon + β·KL on CIFAR-10
def train(cfg: TrainConfig):
    set_seed(cfg.seed)
    device = cfg.device

    print(f"\n{'='*60}")
    print(f"Starting training on: {device}")
    if device.type == 'cuda':
        print(f"GPU Memory allocated: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")
        print(f"GPU Memory cached: {torch.cuda.memory_reserved(0) / 1024**2:.2f} MB")
    print(f"{'='*60}\n")  # <-- FIXED: Changed from {'='=60} to {'='*60}

    print(f"Training VAE on CIFAR-10 with latent_dim={cfg.latent_dim}, beta={cfg.beta}")

    train_loader, test_loader = build_dataloaders(cfg.batch_size, cfg.num_workers, device)

    model = VAE(latent_dim=cfg.latent_dim).to(device)
    # Store beta in model for visualization purposes
    model.beta = cfg.beta

    # Print model size
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")

    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr)

    train_losses, recon_losses, kl_losses = [], [], []
    val_losses = []

    for epoch in range(1, cfg.epochs + 1):
        # Training
        model.train()
        total_loss = 0.0
        total_recon = 0.0
        total_kl = 0.0
        count = 0

        for x, _ in train_loader:
            x = x.to(device, non_blocking=True if device.type == 'cuda' else False)
            x_hat, mu, logvar = model(x)
            loss, recon, kl = vae_loss(x, x_hat, mu, logvar, beta=cfg.beta)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

            bs = x.size(0)
            total_loss += loss.item() * bs
            total_recon += recon.item() * bs
            total_kl += kl.item() * bs
            count += bs

        epoch_loss = total_loss / count
        epoch_recon = total_recon / count
        epoch_kl = total_kl / count

        train_losses.append(epoch_loss)
        recon_losses.append(epoch_recon)
        kl_losses.append(epoch_kl)

        # Validation
        model.eval()
        val_loss = 0.0
        val_count = 0
        with torch.no_grad():
            for x, _ in test_loader:
                x = x.to(device, non_blocking=True if device.type == 'cuda' else False)
                x_hat, mu, logvar = model(x)
                loss, _, _ = vae_loss(x, x_hat, mu, logvar, beta=cfg.beta)
                val_loss += loss.item() * x.size(0)
                val_count += x.size(0)
        val_loss /= val_count
        val_losses.append(val_loss)

        # Print progress with GPU memory info if available
        if device.type == 'cuda':
            gpu_memory = torch.cuda.memory_allocated(0) / 1024**2
            print(f"Epoch {epoch:03d}/{cfg.epochs} | Train Loss: {epoch_loss:.3f} | Recon: {epoch_recon:.3f} | KL: {epoch_kl:.3f} | Val Loss: {val_loss:.3f} | GPU Mem: {gpu_memory:.1f}MB")
        else:
            print(f"Epoch {epoch:03d}/{cfg.epochs} | Train Loss: {epoch_loss:.3f} | Recon: {epoch_recon:.3f} | KL: {epoch_kl:.3f} | Val Loss: {val_loss:.3f}")

        # Save qualitative samples every few epochs
        if epoch == 1 or epoch % max(1, cfg.epochs // 10) == 0 or epoch == cfg.epochs:
            # Random samples
            sample_path = os.path.join(cfg.out_dir, f"beta{cfg.beta}", f"samples_ep{epoch}.png")
            os.makedirs(os.path.dirname(sample_path), exist_ok=True)
            sample_random_grid(model, device, cfg.latent_dim, sample_path, n=16)

            # Reconstructions
            recon_path = os.path.join(cfg.out_dir, f"beta{cfg.beta}", f"reconstructions_ep{epoch}.png")
            plot_reconstructions(model, test_loader, device, recon_path, num_images=8)

    # Save final artifacts
    os.makedirs(cfg.out_dir, exist_ok=True)

    # Training curves
    curves_path = os.path.join(cfg.out_dir, f"beta{cfg.beta}", f"training_curves.png")
    os.makedirs(os.path.dirname(curves_path), exist_ok=True)
    plot_curves(train_losses, recon_losses, kl_losses, curves_path, cfg.beta)

    # Final sample grid
    final_grid = os.path.join(cfg.out_dir, f"beta{cfg.beta}", "final_random_samples.png")
    sample_random_grid(model, device, cfg.latent_dim, final_grid, n=16)

    # Interpolation using real images
    interp_path = os.path.join(cfg.out_dir, f"beta{cfg.beta}", "interpolation.png")
    interpolate_latents(model, device, cfg.latent_dim, interp_path, steps=10, loader=test_loader)

    # Save model weights
    torch.save({
        'model_state_dict': model.state_dict(),
        'cfg': cfg.__dict__,
        'train_losses': train_losses,
        'val_losses': val_losses,
    }, os.path.join(cfg.out_dir, f"beta{cfg.beta}", "model.pt"))

    print(f"\n Training completed! Results saved in {cfg.out_dir}/beta{cfg.beta}/")

    return model, train_losses, recon_losses, kl_losses, val_losses


def parse_args() -> TrainConfig:
    parser = argparse.ArgumentParser(description="VAE/β-VAE on CIFAR-10")
    parser.add_argument('--epochs', type=int, default=50)
    parser.add_argument('--batch-size', type=int, default=128)
    parser.add_argument('--lr', type=float, default=1e-3)
    parser.add_argument('--latent-dim', type=int, default=64)
    parser.add_argument('--beta', type=float, default=1.0)
    parser.add_argument('--num-workers', type=int, default=2)
    parser.add_argument('--out-dir', type=str, default='outputs')
    parser.add_argument('--seed', type=int, default=42)
    parser.add_argument('--force-cpu', action='store_true', help='Force using CPU even if CUDA is available')
    args = parser.parse_args(args=[]) # Modified: Added args=[] to parse_args() to handle notebook execution

    # Determine device
    if args.force_cpu:
        device = torch.device("cpu")
        print("⚠️  Forced CPU usage (--force-cpu flag)")
    else:
        device = get_device()

    return TrainConfig(
        epochs=args.epochs,
        batch_size=args.batch_size,
        lr=args.lr,
        latent_dim=args.latent_dim,
        beta=args.beta,
        num_workers=args.num_workers,
        device=device,
        out_dir=args.out_dir,
        seed=args.seed,
    )


def compare_beta_values():
    """Run experiments with different beta values for Task 4 analysis"""
    betas = [1.0, 5.0]  # Baseline and higher beta
    results = {}

    for beta in betas:
        print(f"\n{'='*60}")
        print(f"Training with beta = {beta}")
        print(f"{'='*60}")

        cfg = TrainConfig(
            epochs=50,
            batch_size=128,
            lr=1e-3,
            latent_dim=64,
            beta=beta,
            num_workers=2,
            device=DEVICE,  # Use the globally detected device
            out_dir='outputs',
            seed=42
        )

        model, train_losses, recon_losses, kl_losses, val_losses = train(cfg)
        results[beta] = {
            'model': model,
            'train_losses': train_losses,
            'recon_losses': recon_losses,
            'kl_losses': kl_losses,
            'val_losses': val_losses
        }

    # Create comparison plot
    plt.figure(figsize=(15, 5))

    for i, (beta, result) in enumerate(results.items()):
        plt.subplot(1, 2, i+1)
        plt.plot(result['train_losses'], label='Total', linewidth=2)
        plt.plot(result['recon_losses'], label='Recon', linewidth=2)
        plt.plot(result['kl_losses'], label=f'KL (β={beta})', linewidth=2)
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title(f'Training Curves (β={beta})')
        plt.legend()
        plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('outputs/beta_comparison.png', dpi=150)
    plt.close()

    print("\nBeta comparison completed! Results saved in outputs/")

    return results


if __name__ == '__main__':
    import sys

    print(f"\n{'='*60}")
    print("VAE for CIFAR-10 Image Synthesis")
    print(f"{'='*60}\n")

    # If '--compare' flag is passed, run beta comparison
    if len(sys.argv) > 1 and sys.argv[1] == '--compare':
        compare_beta_values()
    else:
        cfg = parse_args()
        train(cfg)

CUDA is not available. Using CPU (training will be slower)

VAE for CIFAR-10 Image Synthesis

CUDA is not available. Using CPU (training will be slower)
🔐 Random seed set to 42

Starting training on: cpu

Training VAE on CIFAR-10 with latent_dim=64, beta=1.0
Model total parameters: 1,430,979
Trainable parameters: 1,430,979
Epoch 001/50 | Train Loss: 1904.070 | Recon: 1880.985 | KL: 23.085 | Val Loss: 1860.272
Epoch 002/50 | Train Loss: 1849.504 | Recon: 1821.376 | KL: 28.127 | Val Loss: 1845.495
Epoch 003/50 | Train Loss: 1841.903 | Recon: 1811.735 | KL: 30.168 | Val Loss: 1839.251
Epoch 004/50 | Train Loss: 1838.146 | Recon: 1806.775 | KL: 31.371 | Val Loss: 1836.478
Epoch 005/50 | Train Loss: 1835.742 | Recon: 1803.337 | KL: 32.405 | Val Loss: 1833.884
Epoch 006/50 | Train Loss: 1833.995 | Recon: 1801.163 | KL: 32.832 | Val Loss: 1833.745
Epoch 007/50 | Train Loss: 1832.700 | Recon: 1799.415 | KL: 33.285 | Val Loss: 1831.437
Epoch 008/50 | Train Loss: 1831.521 | Recon: 1797.868 | KL: